# Agentic AI Investigation Team — Scenario 1: Simple BGP Link Down

This notebook is deliberately thin. All architecture — the evidence graph, adapters,
capabilities, agents, and orchestrator — lives in the `investigation_team` package in
this repo, not in notebook cells. This notebook's only job is:

1. Clone/pull the repo and install the package.
2. Provide credentials as environment variables.
3. Define this scenario's specific config (topology, evidence sources, policy).
4. Run the investigation.

See `src/investigation_team/` for the actual implementation, and the project's
[Medium series](https://medium.com/@icaro_vazquez) for the architecture walkthrough
and build log.

## 0. Project setup — clone/pull the repo

In [1]:
from pathlib import Path
import os
import shutil
import subprocess

from google.colab import drive


# ============================================================
# PROJECT SETTINGS
# ============================================================

REPO_NAME = "agentic-ai-investigation-team"
GITHUB_USER = "icarovazquez"
BRANCH = "main"

DRIVE_MOUNT = Path("/content/drive")
PROJECTS_DIR = DRIVE_MOUNT / "MyDrive" / "Colab Notebooks"
PROJECT_ROOT = PROJECTS_DIR / REPO_NAME

DRIVE_SSH_DIR = DRIVE_MOUNT / "MyDrive" / ".ssh_colab"
DRIVE_PRIVATE_KEY = DRIVE_SSH_DIR / "id_ed25519"
DRIVE_PUBLIC_KEY = DRIVE_SSH_DIR / "id_ed25519.pub"

RUNTIME_SSH_DIR = Path.home() / ".ssh"
RUNTIME_PRIVATE_KEY = RUNTIME_SSH_DIR / "id_ed25519"
RUNTIME_PUBLIC_KEY = RUNTIME_SSH_DIR / "id_ed25519.pub"
SSH_CONFIG = RUNTIME_SSH_DIR / "config"
KNOWN_HOSTS = RUNTIME_SSH_DIR / "known_hosts"

SSH_REPO_URL = f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"


def run_command(command, *, cwd=None, check=True):
    result = subprocess.run(command, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}:\n{' '.join(command)}")
    return result


if not (DRIVE_MOUNT / "MyDrive").exists():
    drive.mount(str(DRIVE_MOUNT))
else:
    print("✓ Google Drive is already mounted.")

DRIVE_SSH_DIR.mkdir(parents=True, exist_ok=True)

if not DRIVE_PRIVATE_KEY.exists():
    raise RuntimeError(
        "No persistent SSH key found in Drive. Run the original project-setup "
        "cell once (see the old monolithic notebook) to generate and register one."
    )

RUNTIME_SSH_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_PRIVATE_KEY, RUNTIME_PRIVATE_KEY)
shutil.copy2(DRIVE_PUBLIC_KEY, RUNTIME_PUBLIC_KEY)
os.chmod(RUNTIME_SSH_DIR, 0o700)
os.chmod(RUNTIME_PRIVATE_KEY, 0o600)
os.chmod(RUNTIME_PUBLIC_KEY, 0o644)

SSH_CONFIG.write_text(
    f"""Host github.com
    HostName github.com
    User git
    IdentityFile {RUNTIME_PRIVATE_KEY}
    IdentitiesOnly yes
"""
)
os.chmod(SSH_CONFIG, 0o600)

keyscan = subprocess.run(["ssh-keyscan", "-t", "ed25519", "github.com"], text=True, capture_output=True, check=True)
KNOWN_HOSTS.write_text(keyscan.stdout)
os.chmod(KNOWN_HOSTS, 0o644)

PROJECTS_DIR.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    print(f"Cloning repository into:\n{PROJECT_ROOT}")
    run_command(["git", "clone", "--branch", BRANCH, "--single-branch", SSH_REPO_URL, str(PROJECT_ROOT)])
else:
    status = run_command(["git", "status", "--porcelain", "--untracked-files=no"], cwd=PROJECT_ROOT)
    if status.stdout.strip():
        print("Local changes detected — skipping git pull so nothing is overwritten.")
    else:
        print(f"Pulling the latest origin/{BRANCH}...")
        run_command(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print(f"\n✓ Project root: {PROJECT_ROOT}")


✓ Google Drive is already mounted.
Pulling the latest origin/main...
Already up to date.
From github.com:icarovazquez/agentic-ai-investigation-team
 * branch            main       -> FETCH_HEAD

✓ Project root: /content/drive/MyDrive/Colab Notebooks/agentic-ai-investigation-team


## 1. Install the package

In [2]:
!pip install -e . --quiet


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for investigation-team (pyproject.toml) ... done


## 2. Credentials

Colab-specific: pull secrets from `userdata` and assign them to plain environment
variables. `investigation_team` itself never imports `google.colab` — this is the
only place that boundary gets crossed, deliberately, so the package stays importable
outside Colab too.

In [3]:
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# Note the userdata secret names here are this project's existing Colab secrets
# (LANGFUSE_PUBLIC_KEY_I_TEAM / LANGFUSE_SECRET_KEY_I_TEAM) -- the package itself
# reads the generic LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY names.
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY_I_TEAM").strip()
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY_I_TEAM").strip()

print("✓ Credentials set as environment variables.")


✓ Credentials set as environment variables.


## 3. Import the package

In [4]:
import pprint

from investigation_team.config import (
    EvaluationConfig,
    EvidenceSourceConfig,
    IncidentSeed,
    InvestigationPolicy,
    NetworkInvestigationConfig,
)
from investigation_team.domain import AccessMode, EvidenceType, RemediationMode, SourceFormat
from investigation_team.evaluation import compute_evaluation
from investigation_team.llm import LLM_USAGE
from investigation_team.observability import check_connection
from investigation_team.orchestrator import run_investigation
from investigation_team.persistence import compute_efficiency_metrics, print_run_summary, save_investigation_result

check_connection()


✅ Connected. Projects: ['agentic-a-investigation-team']


## 4. Scenario 1 config: NIKA simple_bgp link_down

In [5]:
nika_simple_bgp_config = NetworkInvestigationConfig(
    investigation_id="nika-simple-bgp-link-down-001",
    investigation_name="NIKA Simple BGP Link Failure",
    environment_name="nika",
    investigation_type="connectivity_failure",

    scenario_provider="nika",
    scenario_id="simple_bgp",

    incident_seed=IncidentSeed(
        reported_symptom=(
            "Users report connectivity issues between "
            "the two endpoint hosts."
        ),
        incident_start_time="2026-08-07T00:00:00Z",
        reported_entities=[
            "nika:simple_bgp:pc1",
            "nika:simple_bgp:pc2",
        ],
        source="nika",
        initial_severity="high",
    ),

    twin_source_id="nika_simple_bgp_topology",

    evidence_sources=[
        EvidenceSourceConfig(
            source_id="nika_simple_bgp_topology",
            evidence_type=EvidenceType.TOPOLOGY,
            source_format=SourceFormat.API,
            access_mode=AccessMode.API,
            adapter_name="NIKASimpleBGPAdapter",
            connection_parameters={"provider": "nika"},
            required=True,
            metadata={
                "scenario_id": "simple_bgp",
                "problem_id": "link_down",
            },
        ),
        EvidenceSourceConfig(
            source_id="nika_simple_bgp_operational",
            evidence_type=EvidenceType.TELEMETRY,
            source_format=SourceFormat.API,
            access_mode=AccessMode.API,
            adapter_name="NIKASimpleBGPOperationalAdapter",
            connection_parameters={"provider": "nika"},
            required=True,
            metadata={
                "scenario_id": "simple_bgp",
                "problem_id": "link_down",
                "observed_at": "2026-08-07T00:00:01Z",
            },
        ),
        EvidenceSourceConfig(
            source_id="pagerduty_simple_bgp_incident",
            evidence_type=EvidenceType.EVENT,
            source_format=SourceFormat.API,
            access_mode=AccessMode.API,
            adapter_name="PagerDutySimpleBGPIncidentAdapter",
            connection_parameters={"provider": "pagerduty"},
            required=True,
            metadata={
                "scenario_id": "simple_bgp",
                "problem_id": "link_down",
            },
        ),
    ],

    investigation_policy=InvestigationPolicy(
        remediation_mode=RemediationMode.RECOMMEND_ONLY,
        require_human_approval=True,
        minimum_diagnosis_confidence=0.80,
    ),

    evaluation_config=EvaluationConfig(
        ground_truth_root_cause="link_down",
        ground_truth_fault_type="link_failure",
        ground_truth_entity_ids=[
            "nika:simple_bgp:router1",
            "nika:simple_bgp:router2",
        ],
    ),

    metadata={
        "nika_task": "simple_bgp_link_down",
        "integration_stage": "static_topology",
    },
)

nika_simple_bgp_config.validate()
print(nika_simple_bgp_config)


NetworkInvestigationConfig(id='nika-simple-bgp-link-down-001', provider='nika', scenario='simple_bgp', sources=3)


## 5. Run the investigation

In [6]:
usage_start = len(LLM_USAGE)

run_result = run_investigation(
    investigation_config=nika_simple_bgp_config,
)

usage_end = len(LLM_USAGE)

print()
print("=" * 70)
print_run_summary(run_result)


✓ Loaded nika_simple_bgp_topology using NIKASimpleBGPAdapter
✓ Loaded nika_simple_bgp_operational using NIKASimpleBGPOperationalAdapter
✓ Loaded pagerduty_simple_bgp_incident using PagerDutySimpleBGPIncidentAdapter
✓ Evidence loaded: {'entities': 4, 'relationships': 3, 'observations': 5, 'events': 1, 'changes': 0}
llm call completed
✓ Incident framed: Why is connectivity between pc1 and pc2 failing, given that the shortest path traverses router1 and router2, and a vendor alert reports 100% packet loss on the core interconnect link between these routers?
llm call completed
Hypothesis generation LLM call completed
⚠ hypothesis_generator_agent returned malformed structured output. Attempting format repair.
llm call completed

Status: in_progress
Failed stage: hypothesis_generation
Error: Unable to parse agent response as a Python dictionary or JSON object.

JSON parser said: Expecting ',' delimiter: line 82 column 10 (char 6050)
Python parser said: '[' was never closed (<unknown>, line 2)

## 6. Evaluate, persist, and check efficiency

In [ ]:
evaluation_result = compute_evaluation(
    run_result=run_result,
    evaluation_config=nika_simple_bgp_config.evaluation_config,
)
pprint.pprint(evaluation_result.to_dict())

saved_path = save_investigation_result(
    run_result=run_result,
    investigation_config=nika_simple_bgp_config,
)

efficiency = compute_efficiency_metrics(LLM_USAGE[usage_start:usage_end])
pprint.pprint(efficiency.to_dict())
